In [50]:
# Install packages (run first in Colab/Jupyter)
!pip install langchain-groq langchain-core python-dotenv -q

import os
from dotenv import load_dotenv
load_dotenv()

# Your Groq API key
os.environ["GROQ_API_KEY"] = "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"

from langchain_groq import ChatGroq

# Initialize LLM (UPDATED: using current model)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

print("✓ Setup complete!")

✓ Setup complete!


# Basic LLM call

In [51]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

response = llm.invoke([
    HumanMessage(content="Explain machine learning in one sentence.")
])

print(response.content)
# Output: "Machine learning is a subset of AI that enables systems to learn
#          from data and improve automatically without explicit programming."

Machine learning is a subset of artificial intelligence that enables computers to learn from data, make predictions or decisions, and improve their performance over time without being explicitly programmed.


# PromptTemplate usage

In [52]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "You are a {role}. Answer this question about {topic}: {question}"
)

prompt = template.format(
    role="data scientist",
    topic="neural networks",
    question="What is backpropagation?"
)

print("Formatted Prompt:")
print(prompt)

response = llm.invoke([HumanMessage(content=prompt)])
print("\nLLM Response:")
print(response.content)

Formatted Prompt:
You are a data scientist. Answer this question about neural networks: What is backpropagation?

LLM Response:
Backpropagation is a fundamental algorithm used in training artificial neural networks, particularly in the context of supervised learning. It's a process that allows the network to adjust the weights and biases of its connections to minimize the difference between its predictions and the actual output, also known as the error or loss.

Here's a simplified overview of how backpropagation works:

1. **Forward Pass**: The neural network receives an input and propagates it through the network, layer by layer, to produce an output.
2. **Error Calculation**: The actual output is compared to the predicted output, and the error is calculated using a loss function (e.g., mean squared error or cross-entropy).
3. **Backward Pass**: The error is propagated backwards through the network, layer by layer, to calculate the gradients of the loss with respect to each weight an

# Simple Chain

In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Convert this technical concept into a simple joke: {concept}"
)

# Create chain using LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "machine learning algorithms"})
print(result)
# Output: "Why don't ML algorithms get lost? They always find the optimal
#          path through gradient descent!"

Why did the machine learning algorithm go to therapy?

Because it was struggling to learn from its mistakes, but it was always making progress... and sometimes it just got stuck in an infinite loop of self-doubt.

(Note: this joke is a bit of a stretch, but it tries to poke fun at some of the technical concepts associated with machine learning algorithms, such as:

* Training data and learning from mistakes
* Getting stuck in local optima (infinite loops of self-doubt)
* The iterative process of machine learning (progress, but sometimes taking a few steps back))


# Agent with tool

In [57]:
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent

# Define tools
@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiplies two numbers together."""
    return a * b

@tool
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers together."""
    return a + b

tools = [multiply_numbers, add_numbers]

# Create ReAct agent using LangGraph (new standard)
agent = create_react_agent(llm, tools)

# Test the agent
result = agent.invoke({
    "messages": [("user", "What is 25 times 4?")]
})

# Extract the final response
print(result["messages"][-1].content)
# Output: "25 × 4 = 100"

/tmp/ipykernel_16223/1697799740.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


So the answer is 100.


# Memory example

In [59]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Initialize in-memory message history
message_history = ChatMessageHistory()

# Create prompt that includes conversation history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember the conversation context."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])

# Create the chain
chain = prompt | llm

# Wrap with message history for automatic memory management
memory_chain = RunnableWithMessageHistory(
    chain,
    lambda session_id: message_history,
    input_messages_key="input",
    history_messages_key="history"
)

# First interaction
print("User: Hi, I'm Aesha from Bengaluru")
response1 = memory_chain.invoke(
    {"input": "Hi, I'm Aesha from Bengaluru"},
    config={"configurable": {"session_id": "conversation1"}}
)
print(f"Bot: {response1.content}")
# Output: "Nice to meet you, Aesha from Bengaluru. How can I help you?"

# Second interaction - memory remembers context
print("\nUser: What do you know about me?")
response2 = memory_chain.invoke(
    {"input": "What do you know about me?"},
    config={"configurable": {"session_id": "conversation1"}}
)
print(f"Bot: {response2.content}")
# Output: "You're Aesha from Bengaluru! [continues with context-aware response]"

User: Hi, I'm Aesha from Bengaluru
Bot: Nice to meet you, Aesha from Bengaluru. How are you doing today? Is there anything I can help you with or would you like to chat about the city or something else?

User: What do you know about me?
Bot: Aesha is your name, and Bengaluru is your city, but I don't have any more information about you. I'm a helpful assistant, and our conversation just started. If you'd like, you can share more about yourself, and I'll do my best to learn and remember our conversation for any future interactions.
